# 01 - Data Understanding

## Objective
Inspect the raw loan dataset before any transformation: schema, target, missing
values, duplicates, outliers, class balance, and columns that must never be used
as predictors (identifiers or anything only known after the credit decision).

Nothing is modified here -- cleaning happens in notebook 03 / `src/data/preprocessing.py`.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

from src import config
from src.data import preprocessing

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

## Configuration

In [ ]:
RAW_FILE = config.RAW_DATA_FILE
RANDOM_STATE = config.RANDOM_STATE
RAW_FILE

## Data loading

In [ ]:
raw = preprocessing.load_raw_data(RAW_FILE)
print(raw.shape)
raw.head()

## Target identification

The target is detected from a list of candidate names rather than hard-coded.

In [ ]:
TARGET = preprocessing.detect_target(raw)
print("Target column:", TARGET)
raw[TARGET].value_counts(normalize=True).rename("share")

## Schema, missing values, duplicates

In [ ]:
profile = preprocessing.profile_dataframe(raw, TARGET)
print("Rows:", profile["n_rows"], "| Columns:", profile["n_columns"])
print("Duplicate rows:", profile["duplicate_rows"])
print("Missing %:", profile["missing_pct"])
profile["target"]

In [ ]:
raw.dtypes.to_frame("dtype").join(raw.isna().sum().to_frame("missing"))

## Categorical levels

In [ ]:
for column, levels in profile["categorical_levels"].items():
    print(f"{column}: {levels}")

## Outliers and invalid values

Outliers are *reported*, not deleted: an unusually high income or loan amount is
normally a genuine and informative observation. Values that are physically
impossible (e.g. an age of 144 or 123 years of employment) are a different
matter -- they are data-entry errors and get nulled during cleaning.

In [ ]:
preprocessing.outlier_summary(raw)

In [ ]:
for column, (low, high) in config.VALID_RANGES.items():
    if column in raw.columns:
        invalid = raw[(raw[column] < low) | (raw[column] > high)]
        if len(invalid):
            print(f"{column}: {len(invalid)} values outside [{low}, {high}] "
                  f"-> max observed {raw[column].max()}")

## Leakage review

| Column | Known before the credit decision? | Keep as a feature? |
|---|---|---|
| `person_*`, `cb_person_*` | yes -- application and bureau data | yes |
| `loan_amnt`, `loan_intent`, `loan_percent_income` | yes -- requested terms | yes |
| `loan_grade`, `loan_int_rate` | yes -- assigned at underwriting | yes, but they embed the lender's own risk view |
| `loan_status` | no -- the outcome | target |

There is no identifier column in this dataset, and no post-origination field
(payment history, recoveries, delinquency), so target leakage is limited to the
lender-assigned grade/rate. Those are kept because they are genuinely available
at scoring time, and their influence is quantified with SHAP in notebook 06.

In [ ]:
print("Identifier-like columns found:", preprocessing.detect_id_columns(raw))

## Results

* Rows / columns and the exact missing-value counts are printed above.
* The target is binary and imbalanced (minority class = defaults).
* Two numeric columns carry impossible values; two carry missing values.
* No identifier column and no post-outcome column are present.

## Interpretation
The dataset is application-level: one row per loan, no time dimension. That rules
out an out-of-time split later on -- a limitation recorded in the README.

## Conclusion
Proceed to EDA (notebook 02) and then to cleaning + feature engineering, where the
invalid values are nulled and imputed **inside the modelling pipeline** so that no
statistic crosses the train/test boundary.